We want to compute the buoyancy frequency squared using cdftools so need to refactor the coarse-grained datasets contained T and S so they are readable by cdftools.

In [1]:
import xarray as xr

Open coarse grained datasets

In [2]:
domain = 'SO_JET'
directory = f'/gws/nopw/j04/ai4pex/twilder/NEMO_data/DINO/EXP16/features_take2/{domain}/coarsened_data/'
fnT = f'MINT_1d_00610101_00610130_thetao_cg_{domain}.nc'
fnS = f'MINT_1d_00610101_00610130_so_cg_{domain}.nc'
dsT = xr.open_dataset(directory + fnT)
dsS = xr.open_dataset(directory + fnS)
dsT, dsS


/apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/lib/python3.12/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
ERROR 1: PROJ: proj_create_from_database: Open of /apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/share/proj failed
ERROR:fiona._env:PROJ: proj_create_from_database: Open of /apps/jasmin/jaspy/miniforge_envs/jaspy3.12/mf3-25.3.0-3/envs/jaspy3.12-mf3-25.3.0-3-v20250704/share/proj failed


(<xarray.Dataset> Size: 28MB
 Dimensions:  (t: 30, y_c: 54, x_c: 60, z_c: 36)
 Coordinates:
   * t        (t) object 240B 0061-01-02 00:00:00 ... 0061-02-01 00:00:00
     gphit    (y_c, x_c) float64 26kB ...
     glamt    (y_c, x_c) float64 26kB ...
 Dimensions without coordinates: y_c, x_c, z_c
 Data variables:
     thetao   (t, z_c, y_c, x_c) float64 28MB ...,
 <xarray.Dataset> Size: 28MB
 Dimensions:  (t: 30, y_c: 54, x_c: 60, z_c: 36)
 Coordinates:
   * t        (t) object 240B 0061-01-02 00:00:00 ... 0061-02-01 00:00:00
     gphit    (y_c, x_c) float64 26kB ...
     glamt    (y_c, x_c) float64 26kB ...
 Dimensions without coordinates: y_c, x_c, z_c
 Data variables:
     so       (t, z_c, y_c, x_c) float64 28MB ...)

Open gridT from NEMO output

In [3]:
directory = f'/gws/nopw/j04/ai4pex/twilder/NEMO_data/DINO/EXP16/production_take2/{domain}/'
fn = f'MINT_1d_00610101_00610130_grid_T_{domain}.nc'
ds = xr.open_dataset(directory + fn)
ds

<xarray.Dataset> Size: 451MB
Dimensions:              (deptht: 36, axis_nbounds: 2, y: 216, x: 238,
                          time_counter: 30)
Coordinates:
  * deptht               (deptht) float32 144B 5.034 15.32 ... 4.253e+03
    nav_lat              (y, x) float32 206kB ...
    nav_lon              (y, x) float32 206kB ...
  * time_counter         (time_counter) object 240B 0061-01-02 00:00:00 ... 0...
    time_instant         (time_counter) object 240B ...
Dimensions without coordinates: axis_nbounds, y, x
Data variables:
    deptht_bounds        (deptht, axis_nbounds) float32 288B ...
    so                   (time_counter, deptht, y, x) float32 222MB ...
    thetao               (time_counter, deptht, y, x) float32 222MB ...
    time_counter_bounds  (time_counter, axis_nbounds) object 480B ...
    time_instant_bounds  (time_counter, axis_nbounds) object 480B ...
    zos                  (time_counter, y, x) float32 6MB ...
Attributes:
    name:         MINT_1d_00610101_00610130_grid_T
    description:  ocean T grid variables
    title:        ocean T grid variables
    Conventions:  CF-1.6
    timeStamp:    2026-Jan-30 20:45:01 GMT
    uuid:         b8cfedf5-b8cf-46de-80b5-82948d2bad4d
    history:      Mon Feb  2 11:01:32 2026: ncks -d x,41,278 -d y,726,941 /gw...
    NCO:          netCDF Operators version 5.3.3 (Homepage = http://nco.sf.ne...

Now refactor so coarse grained datasets are in same format as native NEMO grid. The following code is generated using a GPT, needed something quick.

In [4]:
# Inspect dimensions and coordinates of all three datasets
print("=== ds (template from NEMO output) ===")
print("Dims:", dict(ds.sizes))
print("Coords:", list(ds.coords))
print()
print("=== dsT (coarse-grained T) ===")
print("Dims:", dict(dsT.sizes))
print("Coords:", list(dsT.coords))
print()
print("=== dsS (coarse-grained S) ===")
print("Dims:", dict(dsS.sizes))
print("Coords:", list(dsS.coords))

=== ds (template from NEMO output) ===
Dims: {'deptht': 36, 'axis_nbounds': 2, 'y': 216, 'x': 238, 'time_counter': 30}
Coords: ['deptht', 'nav_lat', 'nav_lon', 'time_counter', 'time_instant']

=== dsT (coarse-grained T) ===
Dims: {'t': 30, 'y_c': 54, 'x_c': 60, 'z_c': 36}
Coords: ['t', 'gphit', 'glamt']

=== dsS (coarse-grained S) ===
Dims: {'t': 30, 'y_c': 54, 'x_c': 60, 'z_c': 36}
Coords: ['t', 'gphit', 'glamt']


In [8]:
def reformat_to_template(source_ds, template_ds, var_name):
    """
    Reformat source_ds so that the variable `var_name` has the same
    dimensions and coordinates as template_ds.
    """
    # --- Step 1: Build dimension rename mapping ---
    # Map source dim names to template dim names based on known aliases
    alias_groups = [
        {"x", "x_c", "x_f", "lon", "longitude"},
        {"y", "y_c", "y_f", "lat", "latitude"},
        {"z", "z_c", "depth", "deptht", "lev", "level"},
        {"t", "time", "time_counter"},
    ]

    dim_rename = {}
    for src_dim in source_ds.dims:
        if src_dim in template_ds.dims:
            continue  # already matches
        # Find alias group for this source dim
        for group in alias_groups:
            if src_dim in group:
                # Find the template dim in the same alias group with matching size
                for tgt_dim in template_ds.dims:
                    if tgt_dim in group and template_ds.sizes[tgt_dim] == source_ds.sizes[src_dim]:
                        dim_rename[src_dim] = tgt_dim
                        break
                break

    # Rename dimensions to match template
    renamed_ds = source_ds.rename(dim_rename)

    # --- Step 2: Extract data variable with explicit dim ordering ---
    data = renamed_ds[var_name]
    target_order = ("time_counter", "deptht", "y", "x")
    # Use target dims that exist in data; append any extras at the end
    ordered_dims = [d for d in target_order if d in data.dims]
    extra_dims = [d for d in data.dims if d not in target_order]
    data = data.transpose(*(ordered_dims + extra_dims))

    # --- Step 3: Build output dataset with template structure ---
    ds_out = xr.Dataset(attrs=template_ds.attrs)
    ds_out[var_name] = data

    # --- Step 4: Copy ALL coordinates from template whose dims exist in output ---
    for coord_name, coord_var in template_ds.coords.items():
        if set(coord_var.dims).issubset(set(ds_out.dims)):
            ds_out[coord_name] = coord_var

    # --- Step 5: Copy bounds variables referenced by coordinates ---
    for coord_name in list(ds_out.coords):
        bounds_name = ds_out[coord_name].attrs.get("bounds")
        if isinstance(bounds_name, str) and bounds_name in template_ds:
            ds_out[bounds_name] = template_ds[bounds_name]

    # --- Step 6: Rename x_c/y_c to x/y if needed ---
    xy_rename = {}
    if "x_c" in ds_out.dims and "x" not in ds_out.dims:
        xy_rename["x_c"] = "x"
    if "y_c" in ds_out.dims and "y" not in ds_out.dims:
        xy_rename["y_c"] = "y"
    if xy_rename:
        ds_out = ds_out.rename(xy_rename)

    # --- Step 7: Rename glamt/gphit to nav_lon/nav_lat if needed ---
    coord_rename = {}
    if "glamt" in ds_out.coords and "nav_lon" not in ds_out.coords:
        coord_rename["glamt"] = "nav_lon"
    if "gphit" in ds_out.coords and "nav_lat" not in ds_out.coords:
        coord_rename["gphit"] = "nav_lat"
    if coord_rename:
        ds_out = ds_out.rename(coord_rename)

    return ds_out


# Identify variable names
t_var = "thetao" if "thetao" in dsT.data_vars else list(dsT.data_vars)[0]
s_var = "so" if "so" in dsS.data_vars else list(dsS.data_vars)[0]

# Reformat coarse-grained datasets to match NEMO template
ds_T_fmt = reformat_to_template(dsT, ds, var_name=t_var)
ds_S_fmt = reformat_to_template(dsS, ds, var_name=s_var)

# Verify structure matches
print("Template dims:  ", dict(ds.sizes))
print("T formatted dims:", dict(ds_T_fmt.sizes))
print("S formatted dims:", dict(ds_S_fmt.sizes))
print()
print("thetao dim order:", ds_T_fmt[t_var].dims)
print("so dim order:    ", ds_S_fmt[s_var].dims)
print()
print("Template coords:  ", sorted(ds.coords))
print("T formatted coords:", sorted(ds_T_fmt.coords))
print("S formatted coords:", sorted(ds_S_fmt.coords))
print()

ds_T_fmt, ds_S_fmt

Template dims:   {'deptht': 36, 'axis_nbounds': 2, 'y': 216, 'x': 238, 'time_counter': 30}
T formatted dims: {'time_counter': 30, 'y': 54, 'x': 60, 'deptht': 36, 'axis_nbounds': 2}
S formatted dims: {'time_counter': 30, 'y': 54, 'x': 60, 'deptht': 36, 'axis_nbounds': 2}

thetao dim order: ('time_counter', 'deptht', 'y', 'x')
so dim order:     ('time_counter', 'deptht', 'y', 'x')

Template coords:   ['deptht', 'nav_lat', 'nav_lon', 'time_counter', 'time_instant']
T formatted coords: ['deptht', 'nav_lat', 'nav_lon', 'time_counter', 'time_instant']
S formatted coords: ['deptht', 'nav_lat', 'nav_lon', 'time_counter', 'time_instant']



(<xarray.Dataset> Size: 28MB
 Dimensions:              (time_counter: 30, y: 54, x: 60, deptht: 36,
                           axis_nbounds: 2)
 Coordinates:
   * time_counter         (time_counter) object 240B 0061-01-02 00:00:00 ... 0...
     nav_lat              (y, x) float64 26kB ...
     nav_lon              (y, x) float64 26kB ...
   * deptht               (deptht) float32 144B 5.034 15.32 ... 4.253e+03
     time_instant         (time_counter) object 240B ...
 Dimensions without coordinates: y, x, axis_nbounds
 Data variables:
     thetao               (time_counter, deptht, y, x) float64 28MB ...
     time_counter_bounds  (time_counter, axis_nbounds) object 480B ...
     deptht_bounds        (deptht, axis_nbounds) float32 288B ...
     time_instant_bounds  (time_counter, axis_nbounds) object 480B ...
 Attributes:
     name:         MINT_1d_00610101_00610130_grid_T
     description:  ocean T grid variables
     title:        ocean T grid variables
     Conventions:  CF-1.6
     

Now save to file

In [10]:
encoding = {"time_counter": {"unlimited": True}}

ds_S_fmt.to_netcdf(
    f'/gws/nopw/j04/ai4pex/twilder/NEMO_data/DINO/EXP16/features_take2/{domain}/coarsened_data/MINT_1d_00610101_00610130_so_cg_SO_JET_cdftools.nc',
    unlimited_dims=["time_counter"],
)
ds_T_fmt.to_netcdf(
    f'/gws/nopw/j04/ai4pex/twilder/NEMO_data/DINO/EXP16/features_take2/{domain}/coarsened_data/MINT_1d_00610101_00610130_thetao_cg_SO_JET_cdftools.nc',
    unlimited_dims=["time_counter"],
)